**INIZIALIZZAZIONE**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.random import default_rng
from scipy.optimize import curve_fit

**COSTRUZIONE DEL MODELLO**

Creiamo l'array delle posizioni, i frammenti della funzione mu(x) e la funzione mu(x). Verifichiamo che essa sia a valori finiti, simmetrica rispetto all'asse y e non negativa. Effettuiamo una estrazione uniforme tra i punto -5 e 5, rappresentiamo i valori della funzione in corrispondenza dei punti trovati, la funzione continua, il fondo B e l'inviluppo superiore.

In [ ]:


def E(x, a) :
    if np.all(x==0) : return 1 
    return np.sinc(a*x/np.pi)**2

def C(x, d) :
    return np.cos(d*x)**2

#funzione conteggi attesi
def mu(x, B, I, a, d ) :
    return B + I*E(x, a)*C(x,d)

points=np.linspace(-5., 5., 161) #valori in x
print(len(points))
B, I, a, d = 15, 500, 0.9, 5.2 
mu_points=mu(points, B, I, a, d) #valori attesi

x=np.linspace(-5,5, 5000)
y=mu(x,B,I,a,d)
y_inviluppo=B+I*E(x,a)

#is finite
if np.sum(np.isfinite(mu_points))/len(points) == 1 : 
    print("Function is finite.")
else : print("Function has singularity points.")

#simmetry
if  np.array_equal(mu_points, mu_points[::-1]) :
    print("Function is simmetric.")
else : print("Function is not simmetric.")

#positivity
if np.sum(mu_points>0)/len(points) == 1 : 
    print("Function is positive.")
else : print("Function has negative points.")


fig, (ax_top, ax_bottom) = plt.subplots(
    2, 1, figsize=(11, 10), sharex=False,
    gridspec_kw={"height_ratios": [3, 1]},
)
ax_top.scatter(points, mu_points, s=15, color="tab:green")
ax_top.set(xlabel="x [mm]", ylabel="mu(x)", ylim=(-10, B+I+10))
ax_top.axvline(0, color="black", lw=1)
ax_top.plot()
ax_bottom.plot(x, y, color="tab:red", lw=2, label="mu(x)")
ax_bottom.plot(x, y_inviluppo, color="tab:blue", lw=2, label="inviluppo")
ax_bottom.set(xlabel="x [mm]", ylabel="mu(x)")
ax_bottom.axhline(B, color="green", lw=1)
ax_bottom.legend()
plt.tight_layout()
plt.show()

**SIMULAZIONE DI UN'ACQUISIZIONE**

Inizializziamo un generatore di conteggi poissoniana e simuliamo un pseudoesperimento estraendo secondo una poissoniana con parametro valore atteso secondo mu(x). Calcoliamo i conteggi ristrengo il dominio in |x| < 1,2 mm. Rappresentiamo in un grafico la funziomne, i valori simulati e le barre di errore.

In [ ]:
rng=np.random.default_rng(67)
counts=rng.poisson(mu_points)
errors=np.sqrt(counts)
errors[errors==0]=1

bool= np.abs(points)<=1.2
N_c=np.sum(counts[bool==1])
Mu_c=np.sum(mu_points[bool==1])
diff=(N_c-Mu_c)

print("N_c =", N_c)
print("Mu_c =", Mu_c)
print("N_c-Mu_c = ", diff)
print("(|N_c-Mu_c|)/sqrt(Mu_c) =", diff/np.sqrt(Mu_c)) #differenza compatibile con una fluttuazione statistica (circa 1 dev standard)


fig, ax=plt.subplots(figsize=(11., 6.2))
ax.plot(x, y, color="tab:red", lw=2, label="simulazione")
ax.errorbar(points, counts, yerr=errors, fmt='o' ,label="errori poissoniani",capsize=2, color="tab:blue")
ax.legend()


**DISTRIBUZIONE DELLO STIMATORE**

Ripetiamo M=2000 volte l'acquisizione, per ogni pseudoesperimento calcoliamo i conteggi (in dominio ristretto come al punto 2) ottenendo una distribuzione dei conteggi, determiniamo la sua media, deviazione standard, deviazione standard della media e scarto tra media dei conteggi e valore atteso relativo alla deviazione standard della media. Infine grafichiamo la distribuzione dei conteggi.

In [ ]:
M=2000
pseudo= np.zeros((M,) + np.shape(counts), dtype=counts.dtype)
N_c_pseudo=np.zeros(M)
for i in range(M) :
    counts=rng.poisson(mu_points)
    pseudo[i]=counts
    N_c_pseudo[i]=np.sum(counts[bool==1])

N_c_mean=np.mean(N_c_pseudo)
N_c_std=np.std(N_c_pseudo, ddof=1)
N_c_mean_std=N_c_std/np.sqrt(2000)
print("Mean of N_c =", N_c_mean)
print("Standard deviation of N_c =", N_c_std)
print("Standard deviation of N_c mean =", N_c_mean_std)
print("(N_c mean - Mu_c) / Std of N_c mean =", (N_c_mean - Mu_c)/N_c_mean_std)
print("Standard deviation of N_c mean / sqrt(Mu_c) =", N_c_mean_std/np.sqrt(Mu_c))

fig, ax = plt.subplots(figsize=(15., 9.))
ax.set_title("Distribution of N_c")
ax.hist(N_c_pseudo, bins=50, alpha=0.75)
ax.axvline(Mu_c, linestyle="--", color = "tab:red" , linewidth=2, label="Mu_c true")
ax.axvline(N_c_mean, linestyle="--", color = "tab:green" , linewidth=2, label="N_c mean")
ax.legend()


